[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/himanshu231204/pytorch-mastery/blob/main/04_llm_finetuning/quantization.ipynb)

# 06 . Quantization (int8, int4, per-channel, block-wise)

## Concept — Quantization (revision notes)

**What it is**
- Representing weights (and sometimes activations) with fewer bits - e.g. 8-bit or 4-bit integers instead of 32/16-bit floats - plus a floating-point **scale** (and optionally a **zero-point**) that maps integers back to real values: `x ≈ scale * (q - zero_point)`.

**Why it matters**
- LLM inference is memory-bandwidth bound: a 7B model is 28 GB in fp32 but ~3.5 GB in 4-bit. Smaller weights mean fitting on smaller GPUs/CPUs, faster loading, and often higher tokens/sec. QLoRA-style fine-tuning also relies on 4-bit base weights.

**When to use**
- Post-training quantization (PTQ) for deployment: int8 is usually nearly lossless; int4 needs care (block-wise scales, outlier handling). Use quantization-aware training only if PTQ quality is not good enough.

**Math & intuition**
- Symmetric (absmax): `scale = max|x| / (2^(b-1)-1)`, `q = round(x/scale)`. Zero maps exactly to 0.
- Asymmetric (zero-point): `scale = (max-min)/(2^b-1)`, `zp = round(-min/scale)`, `q = round(x/scale)+zp`. Uses the full integer range for skewed data (e.g. post-ReLU activations).
- Rounding error is roughly uniform in `[-scale/2, scale/2]`, so its std is `scale/sqrt(12)`; each extra bit halves the scale, i.e. roughly +6 dB of signal-to-noise ratio.
- A **single outlier** inflates `max|x|` and therefore the scale for every other value; finer granularity (per-channel, per-block) confines the damage. That is the central trade-off: granularity vs. the storage overhead of extra scales.

Everything below is plain PyTorch on synthetic tensors and a tiny model (no downloads).

### 1. Symmetric absmax int8 quantization

**What this cell does (plain language):** we write the simplest quantizer: find the largest absolute value, divide so that value maps to 127, round to integers, store as `int8`, and dequantize by multiplying by the scale again. We look at the maximum and mean error and confirm it is bounded by half a quantization step.

In [1]:
import torch
torch.set_num_threads(1)  # tiny models: multi-threading overhead only slows them down (and keeps runs deterministic)
import math
torch.manual_seed(0)

def quant_absmax(x, bits=8, dim=None):
    # symmetric quantization. dim=None -> one scale for the whole tensor,
    # dim=k -> one scale per slice, reducing over dimension k (kept for broadcasting)
    qmax = 2 ** (bits - 1) - 1                                   # 127 for int8, 7 for int4
    amax = x.abs().max() if dim is None else x.abs().amax(dim=dim, keepdim=True)
    scale = amax.clamp(min=1e-12) / qmax
    q = torch.round(x / scale).clamp(-qmax - 1, qmax).to(torch.int8)
    return q, scale

def dequant(q, scale): return q.float() * scale

x = torch.randn(1000)
q, s = quant_absmax(x)
x_hat = dequant(q, s)
print("scale:", s.item(), "| q dtype:", q.dtype, "| q range:", q.min().item(), q.max().item())
print("max abs error:", (x - x_hat).abs().max().item(), "| scale/2 =", s.item() / 2)
print("mean abs error:", (x - x_hat).abs().mean().item())


scale: 0.03229522332549095 | q dtype: torch.int8 | q range: -98 127
max abs error: 0.016114890575408936 | scale/2 = 0.016147611662745476
mean abs error: 0.008114754222333431


### 2. Asymmetric (zero-point) quantization for skewed data

**What this cell does (plain language):** real activations after ReLU are all non-negative, so a symmetric quantizer wastes half of its integer range on negative numbers that never occur. The zero-point variant shifts the range so `min -> 0` and `max -> 255` (uint8). We compare both on skewed data. Note that a zero-point quantizer can still represent real 0 exactly - important for padding and ReLU zeros.

In [2]:
def quant_affine(x, bits=8):
    qmin, qmax = 0, 2 ** bits - 1
    lo, hi = x.min().clamp(max=0.0), x.max().clamp(min=0.0)       # range must include 0
    scale = (hi - lo).clamp(min=1e-12) / (qmax - qmin)
    zp = torch.round(qmin - lo / scale).clamp(qmin, qmax)
    q = torch.clamp(torch.round(x / scale) + zp, qmin, qmax).to(torch.uint8)
    return q, scale, zp
def dequant_affine(q, scale, zp): return (q.float() - zp) * scale

act = torch.relu(torch.randn(10000) * 2 + 0.5)                    # skewed, non-negative like post-ReLU activations
qs, ss = quant_absmax(act)
qa, sa, zp = quant_affine(act)
mse = lambda a, b: ((a - b) ** 2).mean().item()
print(f"symmetric int8  : scale {ss.item():.5f}  MSE {mse(act, dequant(qs, ss)):.2e}")
print(f"asymmetric uint8: scale {sa.item():.5f}  MSE {mse(act, dequant_affine(qa, sa, zp)):.2e}  (zero-point {int(zp.item())})")
print("real 0.0 round-trips to:", dequant_affine(*quant_affine(torch.tensor([0.0, 1.0, 5.0])))[0].item())


symmetric int8  : scale 0.05960  MSE 1.75e-04
asymmetric uint8: scale 0.02968  MSE 4.28e-05  (zero-point 0)
real 0.0 round-trips to: 0.0


### 3. Error vs. number of bits

**What this cell does (plain language):** we quantize the same Gaussian tensor with 2 to 8 bits and measure the signal-to-noise ratio (in dB). Theory says every extra bit halves the step size and quadruples SNR - about +6 dB. This is the single most useful rule of thumb for judging what a precision cut costs.

In [3]:
def snr_db(x, x_hat): return 10 * torch.log10((x ** 2).mean() / ((x - x_hat) ** 2).mean()).item()

x = torch.randn(100000)
prev = None
for bits in range(2, 9):
    q, s = quant_absmax(x, bits)
    snr = snr_db(x, dequant(q, s))
    print(f"{bits} bits: SNR {snr:6.2f} dB" + (f"  (+{snr - prev:.2f} dB vs previous)" if prev else ""))
    prev = snr


2 bits: SNR   0.54 dB
3 bits: SNR   7.98 dB  (+7.44 dB vs previous)
4 bits: SNR  15.34 dB  (+7.36 dB vs previous)
5 bits: SNR  21.96 dB  (+6.62 dB vs previous)
6 bits: SNR  28.24 dB  (+6.28 dB vs previous)
7 bits: SNR  34.40 dB  (+6.15 dB vs previous)
8 bits: SNR  40.48 dB  (+6.08 dB vs previous)


### 4. Per-tensor vs. per-channel scales

**What this cell does (plain language):** in a weight matrix of shape `(out_features, in_features)` different output rows often have very different magnitudes. A single per-tensor scale is dictated by the largest row, so small rows get crushed into just a few integer levels. Per-channel quantization (one scale per output row) fixes this at the cost of `out_features` extra floats. We build a matrix whose row scales span 100x and compare.

In [4]:
torch.manual_seed(1)
row_scales = torch.logspace(-2, 0, 64)                       # row magnitudes from 0.01 to 1
W = torch.randn(64, 256) * row_scales[:, None]

q_t, s_t = quant_absmax(W, 8, dim=None)                       # per-tensor
q_c, s_c = quant_absmax(W, 8, dim=1)                          # per-output-channel (reduce over in_features)
W_t, W_c = dequant(q_t, s_t), dequant(q_c, s_c)
print("scales stored: per-tensor 1 | per-channel", s_c.numel())
print(f"per-tensor  MSE {mse(W, W_t):.2e}  SNR {snr_db(W, W_t):.1f} dB")
print(f"per-channel MSE {mse(W, W_c):.2e}  SNR {snr_db(W, W_c):.1f} dB")
rel = lambda a, b: ((a - b).norm(dim=1) / a.norm(dim=1))      # relative error per row
print("relative error of the smallest row (0.01 scale): per-tensor", round(rel(W, W_t)[0].item(), 4), "| per-channel", round(rel(W, W_c)[0].item(), 4))


scales stored: per-tensor 1 | per-channel 64
per-tensor  MSE 4.97e-05  SNR 33.6 dB
per-channel MSE 5.06e-06  SNR 43.5 dB
relative error of the smallest row (0.01 scale): per-tensor 0.6544 | per-channel 0.0071


### 5. Outliers: one big value ruins per-tensor scales; clipping helps

**What this cell does (plain language):** LLM activations and weights have rare huge outliers. We inject a single outlier of 100 into a Gaussian tensor: the absmax scale jumps 30x, so ordinary values get only a handful of levels. Percentile clipping (calibrating the range on, say, the 99.9th percentile and saturating the rest) trades a little error on the outlier for much finer resolution everywhere else.

In [5]:
torch.manual_seed(2)
x = torch.randn(10000)
x_out = x.clone(); x_out[0] = 100.0                           # one outlier

def quant_clip(x, bits=8, pct=None):
    qmax = 2 ** (bits - 1) - 1
    amax = x.abs().max() if pct is None else torch.quantile(x.abs(), pct)
    scale = amax / qmax
    q = torch.clamp(torch.round(x / scale), -qmax - 1, qmax)
    return q * scale

normal = slice(1, None)                                       # evaluate error on the non-outlier values
for name, xh in [("clean data, absmax", quant_clip(x)),
                 ("with outlier, absmax", quant_clip(x_out)),
                 ("with outlier, clip@99.9%", quant_clip(x_out, pct=0.999))]:
    ref_ = x if name.startswith("clean") else x_out
    print(f"{name:<26} MSE on normal values {mse(ref_[normal], xh[normal]):.2e} | error on the outlier {abs(ref_[0] - xh[0]).item():.2f}")


clean data, absmax         MSE on normal values 7.10e-05 | error on the outlier 0.01
with outlier, absmax       MSE on normal values 5.11e-02 | error on the outlier 0.00
with outlier, clip@99.9%   MSE on normal values 9.44e-05 | error on the outlier 96.64


### 6. int4 block-wise quantization with nibble packing

**What this cell does (plain language):** 4-bit values only hold 16 levels, so a global scale is hopeless; practical schemes (GPTQ, AWQ, bitsandbytes NF4, GGUF Q4) split the flattened weights into small blocks (32-128 values) with one scale each. We implement it, and also physically **pack two 4-bit values per byte** into a `uint8` tensor - the actual source of the 8x memory saving - then unpack and verify an exact round-trip.

In [6]:
def quant_int4_block(w, block=64):
    flat = w.reshape(-1, block)                                # requires numel % block == 0
    scale = flat.abs().amax(dim=1, keepdim=True).clamp(min=1e-12) / 7
    q = torch.clamp(torch.round(flat / scale), -8, 7).to(torch.int8)
    return q, scale

def pack_int4(q):                                             # q in [-8, 7], shape (nblocks, block)
    u = (q + 8).to(torch.uint8).reshape(-1)                   # shift to 0..15
    return u[0::2] | (u[1::2] << 4)                           # two nibbles per byte

def unpack_int4(packed, shape):
    lo, hi = packed & 0x0F, packed >> 4
    u = torch.stack([lo, hi], dim=1).reshape(-1)
    return (u.to(torch.int8) - 8).reshape(shape)

torch.manual_seed(3)
W = torch.randn(128, 256)
q4, s4 = quant_int4_block(W, block=64)
packed = pack_int4(q4)
assert torch.equal(unpack_int4(packed, q4.shape), q4), "pack/unpack must round-trip exactly"
W4 = (q4.float() * s4).reshape(W.shape)
print("fp32 bytes:", W.numel() * 4, "| packed int4 bytes:", packed.numel(), "+ scales (fp32):", s4.numel() * 4)
print("compression incl. scales: %.2fx" % (W.numel() * 4 / (packed.numel() + s4.numel() * 4)))
print(f"int4 block64 SNR {snr_db(W, W4):.1f} dB (int8 per-tensor was {snr_db(W, dequant(*quant_absmax(W))):.1f} dB)")


fp32 bytes: 131072 | packed int4 bytes: 16384 + scales (fp32): 2048
compression incl. scales: 7.11x
int4 block64 SNR 19.4 dB (int8 per-tensor was 40.4 dB)


### 7. Block size trade-off with outliers (int4)

**What this cell does (plain language):** smaller blocks isolate outliers better (higher quality) but need more scales (more overhead: with fp32 scales and block 16 you pay 2 extra bits per weight!). We sweep block size on a weight tensor with a few large outliers and print SNR and effective bits per weight.

In [7]:
torch.manual_seed(4)
W = torch.randn(256, 256)
idx = torch.randint(0, W.numel(), (40,))
W.view(-1)[idx] *= 15                                        # sprinkle heavy outliers
print(f"{'block':>6} {'SNR dB':>8} {'bits/weight':>12}")
for block in (16, 32, 64, 128, 256, 4096):
    q, s = quant_int4_block(W, block)
    Wh = (q.float() * s).reshape(W.shape)
    print(f"{block:>6} {snr_db(W, Wh):>8.1f} {4 + 32 / block:>12.2f}")


 block   SNR dB  bits/weight
    16     20.7         6.00
    32     19.1         5.00
    64     17.3         4.50
   128     15.5         4.25
   256     13.3         4.12
  4096      3.6         4.01


### 8. Error analysis: is rounding error really uniform with std = scale/sqrt(12)?

**What this cell does (plain language):** we check the theory directly. For well-spread data the rounding error `x - x_hat` should be uniform on `[-scale/2, scale/2]`, so its std is `scale/sqrt(12)`. Then we show the error is NOT uniform when the data is concentrated near zero relative to the scale (most values round to 0), which is what happens in the outlier cases above.

In [8]:
torch.manual_seed(5)
x = torch.randn(200000)
q, s = quant_absmax(x, 8)
err = x - dequant(q, s)
print("measured error std:", round(err.std().item(), 6), "| theory scale/sqrt(12):", round(s.item() / math.sqrt(12), 6))
hist = torch.histc(err, bins=5, min=-s.item() / 2, max=s.item() / 2)
print("error histogram over [-scale/2, scale/2] (flat = uniform):", (hist / hist.sum()).numpy().round(3))

x_o = x.clone(); x_o[0] = 5000.0                              # huge outlier -> huge scale -> almost everything rounds to 0
q, s = quant_absmax(x_o, 8)
print("with outlier: fraction of values quantized to exactly 0:", (q == 0).float().mean().item())


measured error std: 0.010269 | theory scale/sqrt(12): 0.010282
error histogram over [-scale/2, scale/2] (flat = uniform): [0.198 0.201 0.2   0.201 0.2  ]
with outlier: fraction of values quantized to exactly 0: 0.9999949932098389


### 9. Quantize a tiny decoder-only LM and measure the damage

**What this cell does (plain language):** we train a very small causal transformer on synthetic Markov-chain text, then do *fake quantization* (quantize then immediately dequantize) of every `nn.Linear` weight under different schemes and measure the loss on held-out data. This is exactly how PTQ quality is evaluated: same model, perturbed weights, compare perplexity.

In [9]:
import copy, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0)
V, L, d = 32, 24, 64

class TinyLM(nn.Module):
    def __init__(self, layers=2, heads=4):
        super().__init__()
        self.tok, self.pos = nn.Embedding(V, d), nn.Embedding(L, d)
        self.blocks = nn.ModuleList([nn.ModuleDict(dict(
            ln1=nn.LayerNorm(d), qkv=nn.Linear(d, 3 * d), proj=nn.Linear(d, d),
            ln2=nn.LayerNorm(d), fc1=nn.Linear(d, 4 * d), fc2=nn.Linear(4 * d, d))) for _ in range(layers)])
        self.lnf, self.head, self.heads = nn.LayerNorm(d), nn.Linear(d, V), heads
    def forward(self, ids):
        B, T = ids.shape
        h = self.tok(ids) + self.pos(torch.arange(T))
        for b in self.blocks:
            q, k, v = b['qkv'](b['ln1'](h)).chunk(3, -1)
            sp = lambda t: t.view(B, T, self.heads, -1).transpose(1, 2)
            a = F.scaled_dot_product_attention(sp(q), sp(k), sp(v), is_causal=True)
            h = h + b['proj'](a.transpose(1, 2).reshape(B, T, -1))
            h = h + b['fc2'](F.gelu(b['fc1'](b['ln2'](h))))
        return self.head(self.lnf(h))

# synthetic text: a first-order Markov chain with a sparse random transition matrix
trans = torch.softmax(torch.randn(V, V) * 2.5, dim=1)
def gen(n):
    seq = torch.zeros(n, L, dtype=torch.long); seq[:, 0] = torch.randint(0, V, (n,))
    for t in range(1, L): seq[:, t] = torch.multinomial(trans[seq[:, t - 1]], 1).squeeze(-1)
    return seq
train, test = gen(2048), gen(512)

model = TinyLM(); opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
for step in range(300):
    b = train[torch.randint(0, len(train), (64,))]
    loss = F.cross_entropy(model(b[:, :-1]).reshape(-1, V), b[:, 1:].reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step()
model.eval()

@torch.no_grad()
def test_loss(m): return F.cross_entropy(m(test[:, :-1]).reshape(-1, V), test[:, 1:].reshape(-1)).item()
true_ent = -(trans * trans.log()).sum(1).mean().item()
print(f"test loss fp32: {test_loss(model):.4f} nats | (data entropy ~ {true_ent:.3f} nats, the unreachable floor)")

def fake_quantize_model(m, fn):
    m = copy.deepcopy(m)
    with torch.no_grad():
        for mod in m.modules():
            if isinstance(mod, nn.Linear): mod.weight.copy_(fn(mod.weight))
    return m

schemes = {
    "int8 per-tensor":  lambda w: dequant(*quant_absmax(w, 8)),
    "int8 per-channel": lambda w: dequant(*quant_absmax(w, 8, dim=1)),
    "int4 per-tensor":  lambda w: dequant(*quant_absmax(w, 4)),
    "int4 per-channel": lambda w: dequant(*quant_absmax(w, 4, dim=1)),
    "int4 block-32":    lambda w: (lambda q, s: (q.float() * s).reshape(w.shape))(*quant_int4_block(w, 32)),
}
base = test_loss(model)
for name, fn in schemes.items():
    l = test_loss(fake_quantize_model(model, fn))
    print(f"{name:<17} loss {l:.4f}  (delta {l - base:+.4f})  perplexity {math.exp(l):.3f}")


test loss fp32: 1.6379 nats | (data entropy ~ 1.562 nats, the unreachable floor)


int8 per-tensor   loss 1.6381  (delta +0.0002)  perplexity 5.145
int8 per-channel  loss 1.6377  (delta -0.0002)  perplexity 5.143


int4 per-tensor   loss 1.6455  (delta +0.0076)  perplexity 5.183
int4 per-channel  loss 1.6433  (delta +0.0055)  perplexity 5.172


int4 block-32     loss 1.6427  (delta +0.0048)  perplexity 5.169


### 10. A real int8 weight-only Linear layer (memory actually shrinks)

**What this cell does (plain language):** fake quantization only simulates the error. Here we build `QLinear`, which really stores the weight as an `int8` tensor plus per-channel scales and dequantizes on the fly in `forward` (weight-only quantization: the activations stay in float). We swap it into the model, check the outputs match the fake-quantized version exactly, and count the bytes.

In [10]:
class QLinear(nn.Module):
    def __init__(self, lin):
        super().__init__()
        q, s = quant_absmax(lin.weight.data, 8, dim=1)
        self.register_buffer("q", q)                              # int8 weights, shape (out, in)
        self.register_buffer("scale", s)                          # fp32 scales, shape (out, 1)
        self.bias = None if lin.bias is None else nn.Parameter(lin.bias.data.clone())
    def forward(self, x):
        return F.linear(x, self.q.float() * self.scale, self.bias)   # dequantize on the fly

def swap_linears(m):
    for name, child in m.named_children():
        if isinstance(child, nn.Linear): setattr(m, name, QLinear(child))
        else: swap_linears(child)
    return m

def nbytes(m): return sum(p.numel() * p.element_size() for p in list(m.parameters()) + list(m.buffers()))
qmodel = swap_linears(copy.deepcopy(model))
fake = fake_quantize_model(model, schemes["int8 per-channel"])
x = test[:8, :-1]
print("max |logits diff| real-int8 vs fake-int8:", (qmodel(x) - fake(x)).abs().max().item())
print(f"model bytes fp32: {nbytes(model):,} -> int8 weights: {nbytes(qmodel):,}  ({nbytes(model) / nbytes(qmodel):.2f}x smaller)")
print("(embeddings and LayerNorms stay fp32 here, so the ratio is below 4x)")


max |logits diff| real-int8 vs fake-int8: 0.0
model bytes fp32: 423,040 -> int8 weights: 126,720  (3.34x smaller)
(embeddings and LayerNorms stay fp32 here, so the ratio is below 4x)


### 11. PyTorch dynamic quantization (guarded)

**What this cell does (plain language):** PyTorch ships ready-made post-training dynamic quantization: weights are stored as int8 and activations are quantized on the fly per batch, running on optimized CPU kernels. It works on `nn.Linear` layers. Availability depends on the build and CPU backend (and the API lives in `torch.ao`, which is being migrated to `torchao`), so this cell is wrapped in `try/except` and the notebook runs fine if it is unavailable.

In [11]:
import io, warnings
try:
    from torch.ao.quantization import quantize_dynamic
    mlp = nn.Sequential(nn.Linear(256, 512), nn.ReLU(), nn.Linear(512, 256)).eval()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")                       # the torch.ao migration emits deprecation warnings
        qmlp = quantize_dynamic(mlp, {nn.Linear}, dtype=torch.qint8)
    def size_bytes(m):
        buf = io.BytesIO(); torch.save(m.state_dict(), buf); return buf.getbuffer().nbytes
    xin = torch.randn(32, 256)
    with torch.no_grad(): diff = (mlp(xin) - qmlp(xin)).abs().max().item()
    print("layer type after quantization:", type(qmlp[0]).__module__ + "." + type(qmlp[0]).__name__)
    print(f"state_dict size fp32 {size_bytes(mlp):,} B -> dynamic int8 {size_bytes(qmlp):,} B")
    print("max abs output difference vs fp32:", round(diff, 4))
except Exception as e:                                        # ImportError, or no quantized engine on this CPU
    print("dynamic quantization unavailable here:", type(e).__name__, e)
    print("pip install torchao   # the maintained successor of torch.ao.quantization")


layer type after quantization: torch.ao.nn.quantized.dynamic.modules.linear.Linear
state_dict size fp32 1,053,917 B -> dynamic int8 268,813 B
max abs output difference vs fp32: 0.0194


### 12. What quantization buys at LLM scale

**What this cell does (plain language):** a back-of-the-envelope table for a 7-billion-parameter model: weight memory for each precision, including the per-block scale overhead. This is arithmetic only, but it is the calculation you do before deciding what hardware you need.

In [12]:
params = 7e9
rows = [("fp32", 32), ("fp16/bf16", 16), ("int8 (per-channel, negligible overhead)", 8),
        ("int4 block-64 + fp16 scale", 4 + 16 / 64), ("int4 block-32 + fp16 scale", 4 + 16 / 32)]
print(f"{'format':<42}{'bits/weight':>12}{'GB':>8}")
for name, bits in rows:
    print(f"{name:<42}{bits:>12.2f}{params * bits / 8 / 1e9:>8.1f}")


format                                     bits/weight      GB
fp32                                             32.00    28.0
fp16/bf16                                        16.00    14.0
int8 (per-channel, negligible overhead)           8.00     7.0
int4 block-64 + fp16 scale                        4.25     3.7
int4 block-32 + fp16 scale                        4.50     3.9


## Common bugs

- **Dividing by a zero scale**
  A tensor that is all zeros (e.g. a freshly initialised bias or a dead channel) gives `absmax = 0` and `x / 0 = NaN`. Fix: clamp the scale to a tiny minimum, as in `quant_absmax`.

- **Quantizing in the wrong dimension for per-channel**
  For `nn.Linear.weight` of shape `(out, in)` the per-output-channel scale reduces over `dim=1`; reducing over `dim=0` yields per-input-channel scales and gives much worse error. Fix: print the scale shape and match it to the axis you intend.

- **Clamp range off by one**
  Symmetric int8 is `[-128, 127]`, but using `scale = amax/128` lets the max value round to 128 and overflow. Fix: use `qmax = 2^(b-1) - 1`, clamp to the legal integer range before casting.

- **Casting before rounding**
  `x.to(torch.int8)` truncates toward zero instead of rounding, doubling the error and biasing it. Fix: `torch.round(x / scale)` first, clamp, then cast.

- **Integer overflow in int8 arithmetic**
  Multiplying or accumulating `int8` tensors in `int8` wraps around silently. Fix: accumulate in int32 (as real kernels do) or dequantize to float first.

- **Forgetting the zero-point when dequantizing**
  With an asymmetric scheme `x = scale * (q - zp)`; omitting `- zp` shifts every value. Fix: keep `(q, scale, zp)` together and test a round-trip.

- **Judging quality by MSE alone**
  A tiny weight MSE can still hurt a few critical outlier channels badly; layers differ enormously in sensitivity. Fix: evaluate end-task metrics (perplexity, accuracy) before and after, and keep sensitive layers (embeddings, lm_head, norms) in higher precision.

- **Unpacked 'int4' that is really int8**
  Storing 4-bit values in an `int8` tensor gives zero memory saving. Fix: pack two nibbles per byte (section 6) or use a library that does.

## Exercises

Attempt each one in the empty cell right after the question. My full solution is in the cell after that - don't peek until you've tried.

**Exercise 1.** Implement stochastic rounding (round up with probability equal to the fractional part) and show that the *mean* error over many repeats of a constant tensor like `0.3 * scale` is ~0 while deterministic rounding has a constant bias.

In [13]:
# Your attempt for Exercise 1 here


**Solution 1**

In [14]:
# Solution 1
torch.manual_seed(0)
scale = 1.0
x = torch.full((100000,), 0.3 * scale)
det = torch.round(x / scale) * scale
frac = x / scale - torch.floor(x / scale)
sto = (torch.floor(x / scale) + (torch.rand_like(x) < frac).float()) * scale
print("deterministic mean error:", (det - x).mean().item(), "| stochastic mean error:", (sto - x).mean().item())


deterministic mean error: -0.29999998211860657 | stochastic mean error: -0.0015399993862956762


**Exercise 2.** Measure the relative error `||W - W_hat|| / ||W||` of int8 per-channel quantization for matrices with 64, 256 and 1024 columns. Does it depend on the matrix width?

In [15]:
# Your attempt for Exercise 2 here


**Solution 2**

In [16]:
# Solution 2
torch.manual_seed(0)
for n in (64, 256, 1024):
    W = torch.randn(128, n)
    Wh = dequant(*quant_absmax(W, 8, dim=1))
    print(n, "relative error:", ((W - Wh).norm() / W.norm()).item())


64 relative error: 0.0060087526217103004
256 relative error: 0.006839949171990156
1024 relative error: 0.007854997180402279


**Exercise 3.** Quantize only the `head` and `tok` weights of the tiny LM (not the blocks) to int4 per-channel. Which part of the model is more sensitive: the transformer blocks or the embeddings/head?

In [17]:
# Your attempt for Exercise 3 here


**Solution 3**

In [18]:
# Solution 3
def fq_only(m, pred, fn):
    m = copy.deepcopy(m)
    with torch.no_grad():
        for name, p in m.named_parameters():
            if pred(name) and p.dim() == 2: p.copy_(fn(p))
    return m
fn = schemes["int4 per-channel"]
print("blocks only :", test_loss(fq_only(model, lambda n: n.startswith("blocks"), fn)))
print("head only   :", test_loss(fq_only(model, lambda n: n.startswith("head"), fn)))
print("embeddings  :", test_loss(fq_only(model, lambda n: n.startswith("tok"), fn)))
print("baseline    :", test_loss(model))
# In this tiny, easy model all gaps are small (within noise); in real LLMs the embeddings/lm_head are usually kept in higher precision for good reason.


blocks only : 1.6405630111694336


head only   : 1.6403685808181763


embeddings  : 1.6367955207824707


baseline    : 1.637872338294983


**Exercise 4.** Add an `int8` KV-style activation quantization: quantize the *input* of each linear per-tensor on the fly (dynamic activation quantization) in `QLinear` and see how much extra loss it adds.

In [19]:
# Your attempt for Exercise 4 here


**Solution 4**

In [20]:
# Solution 4
class QLinearAct(QLinear):
    def forward(self, x):
        xq, xs = quant_absmax(x, 8)                            # dynamic per-tensor activation scale
        return F.linear(dequant(xq, xs), self.q.float() * self.scale, self.bias)
m2 = copy.deepcopy(model)
def swap_act(m):
    for n, c in m.named_children():
        if isinstance(c, nn.Linear): setattr(m, n, QLinearAct(c))
        else: swap_act(c)
    return m
print("weight-only int8 loss:", test_loss(qmodel), "| weight+activation int8 loss:", test_loss(swap_act(m2)))


weight-only int8 loss: 1.6377087831497192 | weight+activation int8 loss: 1.6378306150436401


**Exercise 5.** Implement NF4-like non-uniform quantization: instead of 16 evenly spaced levels use the 16 quantiles of a standard normal (scaled by block absmax), and compare SNR with uniform int4 on Gaussian data.

In [21]:
# Your attempt for Exercise 5 here


**Solution 5**

In [22]:
# Solution 5
from statistics import NormalDist
nd = NormalDist()
levels = torch.tensor([nd.inv_cdf((i + 0.5) / 16) for i in range(16)])
levels = levels / levels.abs().max()                          # normalise to [-1, 1]
def quant_nf(w, block=64):
    flat = w.reshape(-1, block)
    s = flat.abs().amax(1, keepdim=True)
    idx = (flat / s)[..., None].sub(levels).abs().argmin(-1)   # nearest level
    return (levels[idx] * s).reshape(w.shape)
torch.manual_seed(0)
W = torch.randn(256, 256)
print("uniform int4 SNR:", snr_db(W, (lambda q, s: (q.float() * s).reshape(W.shape))(*quant_int4_block(W, 64))))
print("NF-style  4-bit SNR:", snr_db(W, quant_nf(W)))


uniform int4 SNR: 19.376579523086548
NF-style  4-bit SNR: 20.77611207962036


**Exercise 6.** Calibrate an activation range using a *moving average* of min/max over 20 batches (as PTQ observers do) and compare against using only the first batch.

In [23]:
# Your attempt for Exercise 6 here


**Solution 6**

In [24]:
# Solution 6
torch.manual_seed(0)
batches = [torch.randn(256) * (1 + 0.1 * i) for i in range(20)]
first = batches[0].abs().max()
ema = batches[0].abs().max()
for b in batches[1:]: ema = 0.9 * ema + 0.1 * b.abs().max()
print("first-batch absmax:", first.item(), "| EMA absmax:", ema.item(), "| true overall:", max(b.abs().max() for b in batches).item())


first-batch absmax: 3.4105026721954346 | EMA absmax: 6.800219535827637 | true overall: 10.471933364868164
